# ♟️ Chess Numba — Google Colab LMR Phase 2: 等權 Tail 0.3 vs 線性加權 Tail 0.3 巔峰對決

本 Notebook 專門用於在**獨立的 Google Colab Kernel / 虛擬機（例如 AMD 核心）**上運行 **LMR Phase 2 兩大平滑候選策略之 L2 深度實戰單挑**（300,000 nodes, 600 局, `data/openings.epd`）：

* **挑戰方（Candidate A）**：**線性加權 Tail 0.3**（Linear Weighted Tail 0.3，末端步長更小、賦予更高權重）
* **防守方（Baseline / Candidate B）**：**等權 Tail 0.3**（Polyak-Ruppert Uniform Tail 0.3，大窗口算術均值）

### 🎯 驗證目標
* 深入檢驗「線性加權平均」是否能在消除高頻噪聲的同時，比傳統「等權平均」更精準地鎖定末端最優盆地，為引擎帶來實質 Elo 提升！
* **規格**：固定 300,000 nodes 深度搜尋，每 2 局交換黑白開局，SPRT 0, 3 早停保護。
* **輸出檔案**：`tournament_analysis/l2_match_p2_tail03_linear_vs_uniform.json`


In [1]:
# 1. 檢視 Colab 雲端硬體規格
!lscpu | grep "Model name\|Socket\|Thread\|NUMA\|CPU(s)"
!python3 --version


CPU(s):                                  8
On-line CPU(s) list:                     0-7
Model name:                              AMD EPYC 7B12
Thread(s) per core:                      2
Socket(s):                               1
NUMA node(s):                            1
NUMA node0 CPU(s):                       0-7
Python 3.13.16


In [2]:
# 2. 克隆/拉取最新代碼庫並安裝相依套件
import os
branch = "feat-initial-bitboard-setup"

if not os.path.exists("/content/chess_numba"):
    %cd /content
    !git clone -b {branch} https://github.com/b08203031/chess_numba.git

%cd /content/chess_numba
!git checkout -f {branch}
!rm -f tournament_analysis/l2_match_p2_tail03_linear_vs_uniform.json
!git pull origin {branch}
!pip install -q numba chess numpy


/content
Cloning into 'chess_numba'...
remote: Enumerating objects: 3426, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 3426 (delta 127), reused 119 (delta 96), pack-reused 3262 (from 1)
Receiving objects: 100% (3426/3426), 175.70 MiB | 28.34 MiB/s, done.
Resolving deltas: 100% (2350/2350), done.
/content/chess_numba
Already on 'feat-initial-bitboard-setup'
Your branch is up to date with 'origin/feat-initial-bitboard-setup'.
From https://github.com/b08203031/chess_numba
 * branch            feat-initial-bitboard-setup -> FETCH_HEAD
Already up to date.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 19.1 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done


In [ ]:
# 3. 測試環境並產出/確認兩組候選參數（等權 Tail 0.3 vs 線性加權 Tail 0.3）
import os
import sys
import json
from pathlib import Path

%cd /content/chess_numba
sys.path.insert(0, "/content/chess_numba")

cpu_count = os.cpu_count() or 2
concurrency = max(1, cpu_count - 1 if cpu_count > 2 else cpu_count)
print(f"偵測到 Colab CPU 核心數: {cpu_count}，並行度設為: {concurrency}")

log_file = Path("tune_search/logs/lmr_p2_deep_l1.jsonl")
uniform_json = Path("tune_search/candidate_lmr_p2_tail03_uniform.json")
linear_json = Path("tune_search/candidate_lmr_p2_tail03_linear.json")

# 1. 若本機有 L1 調參日誌，優先以最新日誌產出兩組候選參數
if log_file.exists():
    print("✅ 偵測到本機有 Phase 2 調參日誌，正在產出等權與線性加權候選參數...")
    !python tune_search/summarize_run.py {log_file} --tail 0.3 --weight-mode uniform --out {uniform_json}
    !python tune_search/summarize_run.py {log_file} --tail 0.3 --weight-mode linear --out {linear_json}
else:
    print(f"ℹ️ 本機未偵測到 {log_file}，檢查是否已有同步之候選參數檔...")

# 2. 若候選參數檔不存在（跨機器獨立執行），自動注入 Phase 2 (80輪完賽) 認證參數
FALLBACK_UNIFORM = {
    "LMR_BAD_CAPTURE_BONUS": 1060,
    "LMR_GOOD_CAPTURE_RELIEF": 1046,
    "LMR_KILLER_COUNTER_RELIEF": 921,
    "LMR_NO_TTMOVE_BONUS": 1132,
    "LMR_TTMOVE_REDUCTION": 934,
    "LMR_NOT_IMP_NUM": 208
}
FALLBACK_LINEAR = {
    "LMR_BAD_CAPTURE_BONUS": 1072,
    "LMR_GOOD_CAPTURE_RELIEF": 1047,
    "LMR_KILLER_COUNTER_RELIEF": 924,
    "LMR_NO_TTMOVE_BONUS": 1146,
    "LMR_TTMOVE_REDUCTION": 932,
    "LMR_NOT_IMP_NUM": 210
}

if not uniform_json.exists():
    uniform_json.parent.mkdir(parents=True, exist_ok=True)
    uniform_json.write_text(json.dumps({"averaged_params": FALLBACK_UNIFORM, "weight_mode": "uniform"}, indent=2), encoding="utf-8")
    print(f"📦 已自動建立標準等權候選檔：{uniform_json}")

if not linear_json.exists():
    linear_json.parent.mkdir(parents=True, exist_ok=True)
    linear_json.write_text(json.dumps({"averaged_params": FALLBACK_LINEAR, "weight_mode": "linear"}, indent=2), encoding="utf-8")
    print(f"📦 已自動建立標準線性加權候選檔：{linear_json}")

print("\n" + "=" * 60)
print("=== 挑戰方：線性加權 Tail 0.3 (Linear Weighted) ===")
with open(linear_json, "r", encoding="utf-8") as f:
    data = json.load(f)
    print(json.dumps(data.get("averaged_params", data), indent=2))

print("\n=== 防守方：等權 Tail 0.3 (Uniform Averaging) ===")
with open(uniform_json, "r", encoding="utf-8") as f:
    data = json.load(f)
    print(json.dumps(data.get("averaged_params", data), indent=2))
print("=" * 60)


In [ ]:
# 4. 啟動 L2 世紀單挑對決（等權 Tail 0.3 vs 線性加權 Tail 0.3）
# 規格：300,000 nodes, 1000 局, openings.epd, SPRT 0,3
import os

cpu_count = os.cpu_count() or 2
concurrency = max(1, cpu_count - 1 if cpu_count > 2 else cpu_count)
print(f"啟動 L2 單挑對決，使用並行度: {concurrency}")

!python tools/run_search_param_match.py \
  --params-json tune_search/candidate_lmr_p2_tail03_linear.json \
  --baseline-json tune_search/candidate_lmr_p2_tail03_uniform.json \
  --nodes 300000 \
  --games 1000 \
  --openings data/openings.epd \
  --concurrency {concurrency} \
  --sprt 0,3 \
  --batch-games 50 \
  --adjudicate \
  --out tournament_analysis/l2_match_p2_tail03_linear_vs_uniform.json


In [ ]:
# 5. 世紀單挑戰果深度解析與 Elo 檢驗
import json
from pathlib import Path

out_path = Path("tournament_analysis/l2_match_p2_tail03_linear_vs_uniform.json")
if out_path.exists():
    data = json.loads(out_path.read_text(encoding="utf-8"))
    games = data.get("games", 0)
    score_a = data.get("candidate_score", 0.0)
    elo = data.get("candidate_elo", 0.0)
    err = data.get("candidate_elo_err95", 0.0)
    sprt = data.get("sprt", {})
    decision = sprt.get("decision", "running")
    llr = sprt.get("llr", 0.0)
    
    print("=" * 65)
    print("🏆 LMR Phase 2 平滑法世紀單挑：線性加權 vs 等權 Tail 0.3")
    print("=" * 65)
    print(f"總對局數: {games} 局")
    print(f"線性加權得分率 (Score): {score_a*100:.2f}%")
    print(f"線性加權相對 Elo: {elo:+.2f} ± {err:.2f}")
    print(f"SPRT 判定狀態: {decision.upper()} (LLR: {llr:.3f})")
    print("=" * 65)
    
    if elo > 0:
        print("🎉 結論：線性加權 Tail 0.3 實質佔優！更靠近末端微小步長的平滑成功獲益。")
    else:
        print("🛡️ 結論：等權 Tail 0.3 堅守優勢！大窗口均勻濾波更能抵禦末端隨機擾動。")
else:
    print(f"尚未找到對局結果檔案：{out_path}")
